# 04 — Uplift Modeling

Synthetic uplift modeling with T-Learner and targeting simulation.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Main notebook code

Run this notebook and then we will discuss the output, assumptions and pitfalls.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
RANDOM_STATE=42; rng=np.random.default_rng(RANDOM_STATE)
SYNTHETIC_DIR=Path('../data/synthetic'); PROCESSED_DIR=Path('../data/processed'); SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True); PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
n=80000
u=pd.DataFrame({'user_id':np.arange(n),'age':rng.integers(18,70,n),'prior_orders':rng.poisson(3,n),'app_sessions':rng.poisson(8,n),'price_sensitivity':rng.beta(2,5,n),'loyalty_score':rng.normal(0,1,n)})
u['treatment']=rng.binomial(1,0.5,n)
base=-2.6+0.12*u.prior_orders+0.04*u.app_sessions+0.35*u.loyalty_score-1.0*u.price_sensitivity
u['true_uplift']=0.02+0.12*(u.price_sensitivity>0.45)+0.05*(u.prior_orders<=2)-0.03*(u.loyalty_score>1)
u['p_control']=1/(1+np.exp(-base)); u['p_treatment']=np.clip(u.p_control+u.true_uplift,0.001,0.999)
u['conversion_prob']=np.where(u.treatment==1,u.p_treatment,u.p_control); u['converted']=rng.binomial(1,u.conversion_prob)
features=['age','prior_orders','app_sessions','price_sensitivity','loyalty_score']
train,test=train_test_split(u,test_size=.3,random_state=RANDOM_STATE,stratify=u.treatment)
mt=RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=50,random_state=RANDOM_STATE); mc=RandomForestClassifier(n_estimators=300,max_depth=8,min_samples_leaf=50,random_state=RANDOM_STATE)
mt.fit(train.loc[train.treatment==1,features], train.loc[train.treatment==1,'converted']); mc.fit(train.loc[train.treatment==0,features], train.loc[train.treatment==0,'converted'])
test=test.copy(); test['uplift_score']=mt.predict_proba(test[features])[:,1]-mc.predict_proba(test[features])[:,1]
test['uplift_decile']=pd.qcut(test.uplift_score.rank(method='first'),10,labels=False)+1
dec=test.groupby('uplift_decile').apply(lambda x: pd.Series({'customers':len(x),'treated_conversion':x.loc[x.treatment==1,'converted'].mean(),'control_conversion':x.loc[x.treatment==0,'converted'].mean(),'observed_uplift':x.loc[x.treatment==1,'converted'].mean()-x.loc[x.treatment==0,'converted'].mean(),'avg_true_uplift':x.true_uplift.mean(),'avg_score':x.uplift_score.mean()})).reset_index().sort_values('uplift_decile',ascending=False)
display(dec)
plt.figure(figsize=(8,4)); plt.bar(dec.uplift_decile.astype(str), dec.observed_uplift); plt.title('Observed Uplift by Decile'); plt.show()
u.to_csv(SYNTHETIC_DIR/'synthetic_uplift.csv',index=False); test.to_csv(PROCESSED_DIR/'uplift_scored.csv',index=False)


## Discussion prompts

1. What assumption is strongest here?
2. Which pitfall would break the conclusion?
3. How would you explain this to a non-technical stakeholder?
